# Road Segmentation — Full Training

This notebook trains the selected model (U-Net + ResNet34) on the full DeepGlobe dataset with W&B experiment tracking.

### Designed for Colab disconnects
- **Checkpoints** are saved to Google Drive every epoch (survives runtime resets)
- **W&B** logs metrics to the cloud in real-time (no data loss on disconnect)
- **Resume** from the latest checkpoint by re-running the notebook

### Setup
1. `Runtime > Change runtime type > T4 GPU`
2. Fill in your credentials in cell 3 (Kaggle + W&B)
3. `Runtime > Run all`

---
## 1. Setup

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"
# For private repos:
# REPO_URL = "https://<TOKEN>@github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kagglehub", "wandb", "python-dotenv"], check=True)
    print("Dependencies installed.")
else:
    REPO_DIR = None
    print("Running locally.")

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

if IN_COLAB:
    PROJECT_ROOT = Path(REPO_DIR).resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

---
## 2. Credentials

Set your Kaggle and W&B API keys here. These are only used in this session.

In [ ]:
# ====== FILL IN YOUR CREDENTIALS ======
os.environ["KAGGLE_USERNAME"] = ""  # your Kaggle username
os.environ["KAGGLE_KEY"] = ""       # your Kaggle API key
os.environ["WANDB_API_KEY"] = ""    # from https://wandb.ai/authorize
os.environ["WANDB_PROJECT"] = "road-segmentation"

# Verify
assert os.environ["KAGGLE_USERNAME"], "Set KAGGLE_USERNAME above"
assert os.environ["KAGGLE_KEY"], "Set KAGGLE_KEY above"
assert os.environ["WANDB_API_KEY"], "Set WANDB_API_KEY above"
print("Credentials set.")

---
## 3. Device & Dataset

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR

train_dir = DEEPGLOBE_DATASET_DIR / "train"
if not (train_dir.exists() and any(train_dir.glob("*_sat.*"))):
    print("Downloading dataset...")
    from road_segmentation.data.download import download_dataset
    download_dataset()

sat_count = len(list(train_dir.glob("*_sat.*")))
print(f"Training: {sat_count} image-mask pairs")

---
## 4. Mount Google Drive for Checkpoints

Checkpoints are saved to Drive so they survive Colab disconnects.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/road_segmentation")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR = str(DRIVE_DIR / "checkpoints")
    LOG_DIR = str(DRIVE_DIR / "logs")
    print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
else:
    CHECKPOINT_DIR = "checkpoints"
    LOG_DIR = "logs"
    print(f"Checkpoints: {CHECKPOINT_DIR}")

---
## 5. Configure Training

Load the U-Net + ResNet34 config (winner from model comparison) and set up for full training with W&B.

In [ ]:
from road_segmentation.config import load_config

config = load_config(PROJECT_ROOT / "configs" / "unet_resnet34.yaml")

# Full training settings
config.training.epochs = 50
config.data.batch_size = 8
config.data.num_workers = 2 if IN_COLAB else 4
config.training.mixed_precision = torch.cuda.is_available()

# W&B logging
config.logging.wandb = True
config.logging.save_visualizations_every_n_epochs = 5

# Save to Drive
config.checkpoint.save_dir = CHECKPOINT_DIR
config.logging.log_dir = LOG_DIR

# ====== RESUME FROM CHECKPOINT ======
# On reconnect after disconnect, uncomment and set the path:
# config.checkpoint.resume_from = f"{CHECKPOINT_DIR}/{config.logging.experiment_name}/last.pth"

# Auto-resume: find the latest checkpoint on Drive
ckpt_dir = Path(CHECKPOINT_DIR) / config.logging.experiment_name
last_ckpt = ckpt_dir / "last.pth"
if last_ckpt.exists():
    config.checkpoint.resume_from = str(last_ckpt)
    print(f"Resuming from: {last_ckpt}")
else:
    print("Starting fresh training.")

print(f"\nConfig:")
print(f"  Model: {config.model.arch} + {config.model.encoder_name}")
print(f"  Epochs: {config.training.epochs}")
print(f"  Batch size: {config.data.batch_size}")
print(f"  Image size: {config.data.image_size}")
print(f"  Mixed precision: {config.training.mixed_precision}")
print(f"  W&B: {config.logging.wandb}")
print(f"  Checkpoint dir: {config.checkpoint.save_dir}")

---
## 6. Train

In [ ]:
import random
import numpy as np

from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_train_transform, get_val_transform
from road_segmentation.data.dataset import create_dataloaders
from road_segmentation.models.factory import create_model, count_parameters, freeze_encoder
from road_segmentation.training.losses import create_loss
from road_segmentation.training.trainer import Trainer

# Seed
random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

# Data
pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
train_pairs, val_pairs = split_pairs(
    pairs, val_ratio=config.data.val_split_ratio,
    seed=config.data.val_split_seed,
)

train_transform = get_train_transform(
    config.data.image_size, config.augmentations.train,
    config.normalization.mean, config.normalization.std,
)
val_transform = get_val_transform(
    config.data.image_size, config.normalization.mean, config.normalization.std,
)

train_loader, val_loader = create_dataloaders(
    train_pairs, val_pairs, train_transform, val_transform,
    batch_size=config.data.batch_size,
    num_workers=config.data.num_workers,
    pin_memory=device.type == "cuda",
)
print(f"Data: {len(train_pairs)} train, {len(val_pairs)} val")

# Model
model = create_model(
    arch=config.model.arch,
    encoder_name=config.model.encoder_name,
    encoder_weights=config.model.encoder_weights,
    in_channels=config.model.in_channels,
    classes=config.model.classes,
)
if config.training.freeze_encoder_epochs > 0:
    freeze_encoder(model)

params = count_parameters(model)
print(f"Model: {config.model.arch}+{config.model.encoder_name} | {params['total']/1e6:.1f}M params")

# Train
loss_fn = create_loss(config.loss.type, config.loss.params)
trainer = Trainer(config, model, train_loader, val_loader, loss_fn, device)
best_metrics = trainer.train()

---
## 7. Results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from road_segmentation.training.visualization import plot_training_curves, plot_prediction_samples

# Training curves
plt.style.use("ggplot")
fig = plot_training_curves(trainer.history)
plt.show()

# Best metrics
print("\nBest metrics:")
for k, v in best_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# History table
display(pd.DataFrame(trainer.history).round(4))

In [ ]:
# Prediction visualizations
eval_model = trainer.ema.module if trainer.ema else trainer.model
eval_model.eval()

val_batch = next(iter(val_loader))
images = val_batch["image"][:6].to(device)
masks_gt = val_batch["mask"][:6]

with torch.no_grad():
    logits = eval_model(images)
    probs = torch.sigmoid(logits).cpu()

fig = plot_prediction_samples(
    val_batch["image"][:6], masks_gt, probs,
    config.normalization.mean, config.normalization.std,
)
plt.show()

In [ ]:
print("\nArtifacts:")
print(f"  Best checkpoint: {trainer.ckpt_dir / 'best.pth'}")
print(f"  Last checkpoint: {trainer.ckpt_dir / 'last.pth'}")
print(f"  Logs: {trainer.log_dir}")
if trainer.wandb_run:
    print(f"  W&B dashboard: {trainer.wandb_run.url}")